# FunctorFlow Mini-Sudoku Demo

This notebook ports the course mini-Sudoku example into `FunctorFlow/` as a
small structural-constraint demo. It uses a 4x4 Sudoku board with 2x2 blocks so
the notebook stays lightweight while still showing explicit row, column, and
block constraints.


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
import torch

from FunctorFlow.sudoku_demo import (
    SudokuDemoConfig,
    analyze_sudoku_board,
    base_solution_matrix,
    build_sudoku_constraint_diagram,
    format_grid,
    run_sudoku_demo,
)

diagram = build_sudoku_constraint_diagram()
print(diagram.summary())


## Constraint Diagram

FunctorFlow expresses the Sudoku structure as a diagram with:

- cell digits as the source object
- row, column, and block incidence relations
- left-Kan aggregations into unit-level digit histograms
- duplicate-count morphisms as explicit constraint checks


In [ ]:
solved = base_solution_matrix().reshape(-1)
invalid = solved.clone()
invalid[1] = invalid[0]

solved_report = analyze_sudoku_board(solved)
invalid_report = analyze_sudoku_board(invalid)

{
    "solved_duplicates": solved_report["total_duplicates"],
    "invalid_duplicates": invalid_report["total_duplicates"],
}


In [ ]:
print("Solved board:")
print(format_grid(solved))
print()
print("Invalid board:")
print(format_grid(invalid))


## Smoke-Scale Training Comparison

The course notebook compared a plain Transformer with a GT-style model. This
FunctorFlow port keeps that spirit but uses a very short run so the notebook
remains a practical tutorial artifact.


In [ ]:
config = SudokuDemoConfig(
    train_samples=64,
    val_samples=24,
    batch_size=16,
    epochs=1,
    d_model=32,
    n_heads=4,
    num_layers=1,
    lambda_db=0.02,
    seed=0,
)
result = run_sudoku_demo(config, device=torch.device("cpu"))
{
    name: {
        "val_cell_acc": round(payload["final_val_cell_acc"], 3),
        "val_puzzle_acc": round(payload["final_val_puzzle_acc"], 3),
    }
    for name, payload in result["models"].items()
}


In [ ]:
sample = result["sample"]
print("Puzzle (-1 = blank):")
print(format_grid(sample["puzzle"]))
print()
print("Prediction:")
print(format_grid(sample["prediction"]))
print()
print("Solution:")
print(format_grid(sample["solution"]))
sample["prediction_report"]["total_duplicates"]
